In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develope a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-05-29
Last Modified: 2026-05-29
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
import numpy as np

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# unit and sanity checks!

# STRATEGY COMP
# no balancing?
# pull out neurons above unity
# look at neurons below unity
# > check the fits for different regularization constants
# replicate neurotheory plots

# MOVEMENT
# make a model with just movement and check if r2 is above the unity line
# cv/dr2 with movement

"""--------------------------------------------"""
# one regressor
# add time
"""--------------------------------------------"""

## both

In [ ]:
from sg.models import Encoder

encoder = Encoder(subj_id, sess_id)
encoder.fit_encoder()
encoder.encoder_predict()

In [ ]:
encoder.verify()

In [ ]:
from squiggs.renderers import FitRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR

reg = "DLS"
model = "encoder"

r = FitRenderer(
    y=encoder.robs[:, encoder.reg_idxs[reg]],
    yhat=encoder.robs_predict[model][:, encoder.reg_idxs[reg]],
    rsquared=encoder.scores[model][encoder.reg_idxs[reg]],
    mode="lite",
)

_ = NeuronViewer(
    num_units=encoder.psths[reg].shape[0], render_func=r, fig_dir=FIGURES_DIR
)

In [ ]:
from squiggs.renderers import PETHWeightRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR
from core.data import get_psths_cond, get_choice_ts, get_tavg_sc_cond

"""
drift: 21, 29, 35, 36
response: 16, 25, 26, 28
"""

reg = "DLS"
mode = "response"

sc_tavg = get_tavg_sc_cond(
    encoder.robs[:, encoder.reg_idxs[reg]], encoder.trial_data, cond=mode
)

r = PETHWeightRenderer(
    weights=encoder.encoder.coef_[encoder.reg_idxs[reg], :],
    weight_names=encoder.dm_names,
    robs=encoder.robs[:, encoder.reg_idxs[reg]],
    sc_tavg=sc_tavg,
    event_times=get_choice_ts(encoder.trial_data, mode=mode),
    spike_times=encoder.spike_times[reg],
    peths=get_psths_cond(encoder.psths[reg], encoder.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
)

nv = NeuronViewer(num_units=len(encoder.psths[reg]), render_func=r, fig_dir=FIGURES_DIR)

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "strategy",
        "response_prev",
        "rewarded_prev",
    ],
)
se.plot_cvr2()
se.plot_dr2()

# strategy split

## single session

In [ ]:
from sg.models import StrategyEncoder

encoder_mb = StrategyEncoder(subj_id, sess_id, norm=False, strategy_filter="mb")
encoder_mb.fit_encoder()
encoder_mb.encoder_predict()

encoder_mf = StrategyEncoder(subj_id, sess_id, norm=False, strategy_filter="mf")
encoder_mf.fit_encoder()
encoder_mf.encoder_predict()

In [ ]:
encoder_mb.verify()
encoder_mf.verify()

In [ ]:
encoder_mb.view_fits(reg="DMS")

In [ ]:
encoder_mf.view_fits(reg="DMS")

In [ ]:
# try removing balancing
np.argmin(encoder_mb.scores["encoder"])

In [ ]:
# the neurons in the off-quadrants are interesting
from sklearn.linear_model import LinearRegression

lr = LinearRegression().fit(
    (encoder_mb.scores["encoder"] - encoder_mb.scores["baseline"]).reshape(-1, 1),
    encoder_mf.scores["encoder"] - encoder_mf.scores["baseline"],
)

plt.figure(figsize=(3, 2.5), tight_layout=True)
plt.scatter(
    encoder_mb.scores["encoder"] - encoder_mb.scores["baseline"],
    encoder_mf.scores["encoder"] - encoder_mf.scores["baseline"],
    s=0.5,
    alpha=0.5,
)
plt.plot([-0.8, 1], [-0.8, 1], linewidth=0.5, linestyle="--", color="#666666")
plt.plot(
    [-0.8, 1],
    [lr.coef_[0] * (-0.8) + lr.intercept_, lr.coef_[0] + lr.intercept_],
    linewidth=0.5,
    linestyle="-",
    color="#BA3737",
    label=f"{lr.coef_[0]:.3f}x+{lr.intercept_:.3f}",
)

plt.axhline(y=0, linewidth=0.5, color="k")
plt.axvline(x=0, linewidth=0.5, color="k")

plt.xlabel(r"mb, encoder $r^2$ - baseline $r^2$")
plt.ylabel(r"mf, encoder $r^2$ - baseline $r^2$")
plt.legend()
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression().fit(
    encoder_mb.scores["encoder"].reshape(-1, 1), encoder_mf.scores["encoder"]
)

plt.figure(figsize=(3, 2.5), tight_layout=True)
plt.scatter(
    encoder_mb.scores["encoder"], encoder_mf.scores["encoder"], s=0.5, alpha=0.5
)
plt.plot([-0.5, 1], [-0.5, 1], linewidth=0.5, linestyle="--", color="#666666")
plt.plot(
    [-0.5, 1],
    [lr.coef_[0] * (-0.5) + lr.intercept_, lr.coef_[0] + lr.intercept_],
    linewidth=0.5,
    linestyle="-",
    color="#BA3737",
    label=f"{lr.coef_[0]:.3f}x+{lr.intercept_:.3f}",
)

plt.axhline(y=0, linewidth=0.5, color="k")
plt.axvline(x=0, linewidth=0.5, color="k")

plt.xlabel(r"mb, encoder $r^2$")
plt.ylabel(r"mf, encoder $r^2$")
plt.legend()
plt.show()

### mb > mf

In [ ]:
def mb_gr_mf(encoder_mb, encoder_mf, reg):
    if reg == "all":
        mb = encoder_mb.scores["encoder"]
        mf = encoder_mf.scores["encoder"]

        num_units = encoder_mb.num_units

    else:
        mb = encoder_mb.scores["encoder"][encoder_mb.reg_idxs[reg]]
        mf = encoder_mf.scores["encoder"][encoder_mf.reg_idxs[reg]]

        num_units = encoder_mb.psths[reg].shape[0]

    return np.round((mb > mf).sum() / num_units, 3)


r2_strategy_comp = {}
for reg in ["all", "DLS", "DMS"]:
    r2_strategy_comp[reg] = mb_gr_mf(encoder_mb, encoder_mf, reg)

In [ ]:
r2_strategy_comp

In [ ]:
regs = ["all", "DLS", "DMS"]
fig, axes = plt.subplots(ncols=3, nrows=1, figsize=(4, 2))

for i, ax in enumerate(axes.flat):
    reg = regs[i]
    ax.pie(
        [1 - r2_strategy_comp[reg], r2_strategy_comp[reg]],
        colors=["#9C9C9C", "#AEC3F1"],
        autopct="%.1f%%",
        startangle=90,
    )

In [ ]:
from core.data import subject_ids, session_ids

sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]


def mb_gr_mf(encoder_mb, encoder_mf, reg):
    if reg == "all":
        mb = encoder_mb.scores["encoder"]
        mf = encoder_mf.scores["encoder"]

        num_units = encoder_mb.num_units

    else:
        mb = encoder_mb.scores["encoder"][encoder_mb.reg_idxs[reg]]
        mf = encoder_mf.scores["encoder"][encoder_mf.reg_idxs[reg]]

        num_units = encoder_mb.psths[reg].shape[0]

    return np.round((mb > mf).sum() / num_units, 3)


r2_strategy_comp = {"all": [], "DMS": [], "DLS": []}
p_mbs = []

for i, sess_id in enumerate(sess_ids):
    encoder = Encoder(subj_id, sess_id, norm=False)
    encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb", norm=False)
    encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf", norm=False)

    encoder.get_data()

    try:
        encoder_mb.get_r2()
        encoder_mf.get_r2()
    except RuntimeError:
        continue

    p_mbs.append((encoder.trial_data["strategy"] == 1).mean())

    for reg in ["all", "DMS", "DLS"]:
        r2_strategy_comp[reg].append(mb_gr_mf(encoder_mb, encoder_mf, reg=reg))

p_mbs = np.array(p_mbs)
r2_strategy_comp = {reg: np.array(r2_strategy_comp[reg]) for reg in r2_strategy_comp}

In [ ]:
from sklearn.linear_model import LinearRegression

reg = "DLS"
lr = LinearRegression().fit(p_mbs.reshape(-1, 1), r2_strategy_comp[reg])

plt.figure(tight_layout=True)
plt.scatter(p_mbs, r2_strategy_comp[reg], s=0.5, alpha=0.5)
plt.plot(
    [0, 1],
    [lr.intercept_, lr.coef_[0] + lr.intercept_],
    label=f"{lr.coef_[0]:.3f}x+{lr.intercept_:.3f}",
)

plt.xlabel("p(mb)")
plt.ylabel(rf"mb $r^2$ > mf $r^2$, {reg}")
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(tight_layout=True)

ax.plot(p_mbs, color="#222222")
ax.axhline(y=0.5, color="#666666", linewidth=0.5, linestyle="--")
ax.set_ylabel("p(mb)")

ax2 = ax.twinx()

ax2.plot(r2_strategy_comp["all"], color="#523784", label="all")
ax2.plot(r2_strategy_comp["DMS"], color="#0A581A", label="DMS")
ax2.plot(r2_strategy_comp["DLS"], color="#29C71A", label="DLS")
ax2.set_ylabel(r"p(mb $r^2$ > mf $r^2$)")

fig.legend()

In [ ]:
# correlates with p(mb) between sessions and animals?
# across sessions


def mb_gr_mf(encoder_mb, encoder_mf, reg):
    mb = (encoder_mb.scores["encoder"])[encoder_mb.reg_idxs[reg]]
    mf = (encoder_mf.scores["encoder"])[encoder_mf.reg_idxs[reg]]

    num_units = encoder_mb.psths[reg].shape[1]

    return np.round((mb > mf).sum() / num_units, 3)


mb_gr_mf(encoder_mb, encoder_mf, reg="DLS")

### beta weights

In [ ]:
from squiggs.renderers import FitRenderer
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR

reg = "DLS"

r = FitRenderer(
    y=encoder_mf.robs[:, encoder_mf.reg_idxs[reg]],
    yhat=encoder_mf.robs_predict["encoder"][:, encoder_mb.reg_idxs[reg]],
    mode="lite",
)

_ = NeuronViewer(
    num_units=encoder_mb.psths[reg].shape[1], render_func=r, fig_dir=FIGURES_DIR
)

In [ ]:
from squiggs.renderers import PETHWeightRenderer
from core.data import get_tavg_sc_cond, get_choice_ts, get_psths_cond
import numpy as np

reg = "DLS"
mode = "response"

sc_tavg = get_tavg_sc_cond(
    encoder_mb.robs[:, encoder_mb.reg_idxs[reg]], encoder_mb.trial_data, cond=mode
)

r = PETHWeightRenderer(
    weights=np.hstack(
        (
            encoder_mb.baseline_model.coef_[encoder_mb.reg_idxs[reg], :],
            encoder_mb.encoder.coef_[encoder_mb.reg_idxs[reg], :],
        )
    ),
    weight_names=encoder_mb.dm_names,
    robs=encoder_mb.robs[:, encoder_mb.reg_idxs[reg]],
    sc_tavg=sc_tavg,
    event_times=get_choice_ts(encoder_mb.trial_data, mode=mode),
    spike_times=encoder_mb.spike_times[reg],
    peths=get_psths_cond(encoder_mb.psths[reg], encoder_mb.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
)

nv = NeuronViewer(
    num_units=len(encoder_mb.psths[reg]), render_func=r, fig_dir=FIGURES_DIR
)

In [ ]:
from squiggs.renderers import PETHWeightRenderer

reg = "DLS"
mode = "response"

sc_tavg = get_tavg_sc_cond(
    encoder_mf.robs[:, encoder_mf.reg_idxs[reg]], encoder_mf.trial_data, cond=mode
)

r = PETHWeightRenderer(
    weights=encoder_mf.encoder.coef_[encoder_mf.reg_idxs[reg], :],
    weight_names=encoder_mf.dm_names,
    robs=encoder_mf.robs[:, encoder_mf.reg_idxs[reg]],
    sc_tavg=sc_tavg,
    event_times=get_choice_ts(encoder_mf.trial_data, mode=mode),
    spike_times=encoder_mf.spike_times[reg],
    peths=get_psths_cond(encoder_mf.psths[reg], encoder_mf.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
)

nv = NeuronViewer(
    num_units=len(encoder_mf.psths[reg]), render_func=r, fig_dir=FIGURES_DIR
)

In [ ]:
plt.figure()
plt.imshow(
    encoder_mb.encoder_weights, vmin=-2, vmax=2, cmap="coolwarm"
)  # , aspect='auto')
plt.colorbar()
plt.show()

In [ ]:
encoder_mb.encoder_weights[1, 5:]

In [ ]:
idx = 11
plt.figure()
plt.scatter(encoder_mb.encoder_weights[idx, 5:], encoder_mf.encoder_weights[idx, 5:])
# plt.plot([0, 0.05], [0, 0.05])
plt.show()

In [ ]:
# weight comparison
import numpy as np

scale = 10

regr = "response"
val = "left"

i = np.where(encoder_mb.dm_names == f"{regr}_{val}")[0][0]

plt.figure(tight_layout=True)
plt.scatter(
    encoder_mb.encoder_weights[:, i], encoder_mf.encoder_weights[:, i], s=0.5, alpha=0.5
)

plt.plot([-1, 1.5], [-1, 1.5], color="#666666", linewidth=0.5, linestyle="--")

plt.axhline(y=0, color="k", linewidth=0.5)
plt.axvline(x=0, color="k", linewidth=0.5)
plt.xlabel(f"mb, bweight {regr} {val}")
plt.ylabel(f"mf, bweight {regr} {val}")
plt.legend()
plt.plot()

In [ ]:
import numpy as np

In [ ]:
reg = "DLS"
regr = "response"
val = "right"
regr_idx_mb = np.where(encoder_mb.dm_names == f"{regr}_{val}")[0][0]
regr_idx_mf = np.where(encoder_mf.dm_names == f"{regr}_{val}")[0][0]

coef_mb = encoder_mb.encoder_weights[encoder_mb.reg_idxs[reg], regr_idx_mb]
coef_mf = encoder_mf.encoder_weights[encoder_mf.reg_idxs[reg], regr_idx_mf]

# mb_gr_idxs = np.where(np.abs(coef_mb) > np.abs(coef_mf) * scale)[0]
# mf_gr_idxs = np.where(np.abs(coef_mf) > np.abs(coef_mb) * scale)[0]

coef_diff = coef_mb - coef_mf
idxs = np.flip(
    np.argsort(np.abs(coef_diff))  # / encoder.robs.mean(axis=0)[encoder.reg_idxs[reg]])
)

In [ ]:
plt.figure(tight_layout=True)
plt.hist(coef_diff, bins=np.linspace(-0.2, 0.2, 17))
plt.xlabel(r"mb - mf ($\beta$ response_right)")
plt.ylabel("count")
plt.show()

In [ ]:
reg = "DLS"
idxs = np.where(
    encoder_mb.scores["encoder"][encoder_mb.reg_idxs[reg]]
    > encoder_mf.scores["encoder"][encoder_mf.reg_idxs[reg]]
)[0]
a = np.flip(
    np.argsort(
        encoder_mb.scores["encoder"][encoder_mb.reg_idxs[reg]]
        - encoder_mf.scores["encoder"][encoder_mf.reg_idxs[reg]]
    )
)

In [ ]:
a[-3:]

In [ ]:
# idxs = np.where(coef_mb/coef_mf < 0)[0]
idxs

In [ ]:
from squiggs.renderers import PETHWeightCompRenderer
from squiggs.neuron_viewer import NeuronViewer
from core.data import get_tavg_sc_cond, get_choice_ts, get_psths_cond
from utils.paths import FIGURES_DIR

reg = "DLS"
mode = "response"

sc_tavg_mb = get_tavg_sc_cond(
    encoder_mb.robs[:, encoder_mb.reg_idxs[reg]], encoder_mb.trial_data, cond=mode
)

sc_tavg_mf = get_tavg_sc_cond(
    encoder_mf.robs[:, encoder_mf.reg_idxs[reg]], encoder_mf.trial_data, cond=mode
)

r = PETHWeightCompRenderer(
    weights={
        "mb": encoder_mb.encoder_weights[encoder_mb.reg_idxs[reg], :],
        "mf": encoder_mf.encoder_weights[encoder_mf.reg_idxs[reg], :],
    },
    weight_names=encoder_mb.dm_names,
    robs={
        "mb": encoder_mb.robs[:, encoder_mb.reg_idxs[reg]],
        "mf": encoder_mf.robs[:, encoder_mf.reg_idxs[reg]],
    },
    sc_tavgs={"mb": sc_tavg_mb, "mf": sc_tavg_mf},
    event_times={
        "mb": get_choice_ts(encoder_mb.trial_data, mode=mode),
        "mf": get_choice_ts(encoder_mf.trial_data, mode=mode),
    },
    spike_times=encoder_mb.spike_times[reg],  # == encoder_mf.spike_times
    peths={
        "mb": get_psths_cond(encoder_mb.psths[reg], encoder_mb.trial_data, mode=mode),
        "mf": get_psths_cond(encoder_mf.psths[reg], encoder_mf.trial_data, mode=mode),
    },
    same_ylim=True,
)

_ = NeuronViewer(encoder_mb.psths[reg].shape[0], r, fig_dir=FIGURES_DIR)

## aggregate

In [ ]:
import numpy as np
from core.data import subject_ids, session_ids

sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]
coefs = {"mb": {"DLS": [], "DMS": []}, "mf": {"DLS": [], "DMS": []}}
regressors = np.array(
    [
        "response_left",
        "response_right",
        "rewarded_incorr",
        "rewarded_corr",
    ]
)

for sess_id in sess_ids:
    encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")
    encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf")

    try:
        encoder_mb.fit_encoder()
        encoder_mf.fit_encoder()
    except RuntimeError:
        continue

    # get coefs and separate by region as well
    coefs_mb = encoder_mb.encoder_weights
    coefs_mf = encoder_mf.encoder_weights

    tv_idxs = [
        i for i, dm_name in enumerate(encoder_mb.dm_names) if dm_name in regressors
    ]

    coefs_mb_ = coefs_mb[:, tv_idxs]
    coefs_mf_ = coefs_mf[:, tv_idxs]

    coefs["mb"]["DLS"].extend(coefs_mb_[encoder_mb.reg_idxs["DLS"]])
    coefs["mb"]["DMS"].extend(coefs_mb_[encoder_mb.reg_idxs["DMS"]])
    coefs["mf"]["DLS"].extend(coefs_mf_[encoder_mf.reg_idxs["DLS"]])
    coefs["mf"]["DMS"].extend(coefs_mf_[encoder_mf.reg_idxs["DMS"]])

coefs = {
    strategy: {region: np.array(coefs[strategy][region]) for region in coefs[strategy]}
    for strategy in coefs
}

In [ ]:
# sanity check!

In [ ]:
from sklearn.linear_model import LinearRegression


def plot_bweight_strategy(reg, regr, val):
    regr_idx = np.where(regressors == f"{regr}_{val}")[0]

    lr = LinearRegression().fit(
        coefs["mb"][reg][:, regr_idx], coefs["mf"][reg][:, regr_idx]
    )

    plt.figure(figsize=(2.5, 2), tight_layout=True)
    plt.scatter(
        coefs["mb"][reg][:, regr_idx],
        coefs["mf"][reg][:, regr_idx],
        s=0.5,
        alpha=0.5,
        zorder=2,
    )

    plt.plot([-1.5, 1.5], [-1.5, 1.5], linewidth=0.5, linestyle="--", color="#666666")
    plt.plot(
        [-1.5, 1.5],
        [lr.coef_[0] * (-1.5) + lr.intercept_, lr.coef_[0] * (1.5) + lr.intercept_],
        linewidth=0.5,
        linestyle="-",
        color="#A52424",
        label=f"{lr.coef_[0][0]:.3f}x+{lr.intercept_[0]:.3f}",
    )

    plt.axhline(y=0, color="k", linewidth=0.5)
    plt.axvline(x=0, color="k", linewidth=0.5)

    plt.xlabel(rf"mb $\beta$ {regr}_{val}")
    plt.ylabel(rf"mf $\beta$ {regr}_{val}")

    plt.legend()

    plt.show()


plot_bweight_strategy(reg="DLS", regr="response", val="right")
plot_bweight_strategy(reg="DMS", regr="response", val="left")